# Cómputo del modelo global

El servidor recibe los modelos locales (`models/lmodel_{k}.keras`) y el número de ejemplos con que se entrenó cada uno (`models/lmodel_{k}.json`), y los combina en un único modelo global con la misma arquitectura (`CaffeLeNet.py`).

In [ ]:
import glob
import json
import os

import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report

from CaffeLeNet import build

SEED = 42
EPOCHS = 5       # mismos valores que en local_training.ipynb,
BATCH_SIZE = 64  # para que el modelo centralizado sea comparable

### 1. Carga de los modelos locales

In [ ]:
model_paths = sorted(glob.glob("models/lmodel_*.keras"))
local_models = [tf.keras.models.load_model(path) for path in model_paths]

client_sizes = []
for path in model_paths:
    with open(path.replace(".keras", ".json"), encoding="utf-8") as f:
        client_sizes.append(json.load(f)["n_samples"])

# Todos los modelos locales deben tener la arquitectura del modelo global.
expected_shapes = [w.shape for w in build.build_it().get_weights()]
for path, model in zip(model_paths, local_models):
    assert [w.shape for w in model.get_weights()] == expected_shapes, f"{path}: arquitectura distinta"

print("Modelos locales:", [os.path.basename(path) for path in model_paths])
print("Ejemplos por cliente (n_k):", client_sizes)

### 2. Conjunto de prueba

In [ ]:
_, (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_test = np.expand_dims(x_test / 255.0, -1).astype("float32")


def accuracy(model):
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    return float(np.mean(y_pred == y_test)), y_pred


def evaluate(model, name):
    """Calcula la exactitud y guarda el classification report en reports/{name}.txt."""
    acc, y_pred = accuracy(model)
    report = classification_report(y_test, y_pred, digits=4)
    with open(f"reports/{name}.txt", "w", encoding="utf-8") as f:
        f.write(report)
    print(f"Classification report de {name}:\n{report}")
    return acc

### 3. FedAvg

FedAvg (McMahan et al., 2017) promedia, parámetro por parámetro, los pesos de los $K$ modelos locales, ponderando cada uno por la fracción de datos con que se entrenó:

$$w_{\text{global}} = \sum_{k=1}^{K} \frac{n_k}{n}\, w_k, \qquad n = \sum_{k=1}^{K} n_k.$$

Como las particiones tienen el mismo tamaño, el resultado coincide con el promedio simple; la ponderación se implementa para que el método sea correcto con particiones de cualquier tamaño. FedAvg supone que todos los clientes parten del mismo modelo inicial $w_0$ (ver `SHARED_INIT` en `local_training.ipynb`).

In [ ]:
def fedavg(client_weights, client_sizes):
    """Promedio de los pesos de los clientes, ponderado por n_k / n."""
    coefs = np.asarray(client_sizes, dtype=np.float64)
    coefs /= coefs.sum()
    return [
        sum(c * layer for c, layer in zip(coefs, layers)).astype(np.float32)
        for layers in zip(*client_weights)
    ]


global_model = build.build_it(name="global_fedavg")
global_model.set_weights(fedavg([m.get_weights() for m in local_models], client_sizes))
acc_fedavg = evaluate(global_model, "global_fedavg")
global_model.save("models/global_fedavg.keras")

### 4. Referencias de comparación

- **Modelos locales**: exactitud de cada modelo local en el conjunto de prueba (sus reportes completos están en `reports/lmodel_{k}.txt`).
- **Modelo centralizado**: el mismo modelo entrenado con los 60,000 ejemplos de entrenamiento en un solo dispositivo, con las mismas épocas y tamaño de lote. Es una cota superior de referencia, no un escenario federado: requiere reunir todos los datos en un solo lugar.

In [ ]:
local_acc = {
    os.path.basename(path).removesuffix(".keras"): accuracy(model)[0]
    for path, model in zip(model_paths, local_models)
}

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train / 255.0, -1).astype("float32")

tf.keras.utils.set_random_seed(SEED)
central_model = build.build_it(name="centralizado")
central_model.fit(x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
acc_central = evaluate(central_model, "centralizado")

### 5. Resumen de resultados

In [ ]:
results = {
    **local_acc,
    "promedio_modelos_locales": float(np.mean(list(local_acc.values()))),
    "global_fedavg": acc_fedavg,
    "centralizado": acc_central,
}

with open("reports/resultados.csv", "w", encoding="utf-8") as f:
    f.write("modelo,exactitud\n")
    for name, acc in results.items():
        f.write(f"{name},{acc:.4f}\n")

print(f"{'Modelo':<26}Exactitud (prueba)")
for name, acc in results.items():
    print(f"{name:<26}{acc:.4f}")